# spaCy - A Complete Hands-On Tutorial

Welcome! This notebook is a comprehensive, beginner-friendly guide to [spaCy](https://spacy.io/), one of the most popular and powerful Natural Language Processing (NLP) libraries in Python.

**What is NLP?** Natural Language Processing is a branch of AI that helps computers understand, interpret, and generate human language. Think of it as teaching a computer to read.

**Why spaCy?** Unlike many academic NLP tools, spaCy is designed for *production use*. It's fast, memory-efficient, and comes with pre-trained models that work out of the box.

**Prerequisites:** You should be comfortable with Python basics (variables, loops, functions, classes). No NLP experience needed!

---

## Table of Contents

**Getting Started**
1. [Installation & Setup](#1.-Installation-&-Setup)
2. [The Doc, Token, and Span Objects](#2.-Core-Concepts:-Doc,-Token,-and-Span)

**Core NLP Features**

3. [Tokenization](#3.-Tokenization)
4. [Part-of-Speech (POS) Tagging](#4.-Part-of-Speech-(POS)-Tagging)
5. [Lemmatization](#5.-Lemmatization)
6. [Named Entity Recognition (NER)](#6.-Named-Entity-Recognition-(NER))
7. [Dependency Parsing](#7.-Dependency-Parsing)
8. [Sentence Segmentation](#8.-Sentence-Segmentation)
9. [Morphological Analysis](#9.-Morphological-Analysis)

**Intermediate Features**

10. [Word Vectors & Similarity](#10.-Word-Vectors-&-Similarity)
11. [Rule-Based Matching](#11.-Rule-Based-Matching)
12. [EntityRuler - Custom NER Rules](#12.-EntityRuler---Custom-NER-Rules)
13. [Stop Words & Text Preprocessing](#13.-Stop-Words-&-Text-Preprocessing)
14. [Vocab, Lexemes & StringStore](#14.-Vocab,-Lexemes-&-StringStore)

**Advanced Features**

15. [Processing Pipelines & Performance](#15.-Processing-Pipelines-&-Performance)
16. [Custom Extensions (Token, Span, Doc)](#16.-Custom-Extensions)
17. [Custom Pipeline Components](#17.-Custom-Pipeline-Components)
18. [Retokenization & Merging](#18.-Retokenization-&-Merging)
19. [Serialization - Saving & Loading](#19.-Serialization---Saving-&-Loading)
20. [Multi-Language Support](#20.-Multi-Language-Support)

**Wrap-Up**

21. [Summary & Cheat Sheet](#21.-Summary-&-Cheat-Sheet)

---
# Getting Started
---

## 1. Installation & Setup

spaCy works in two parts:
1. **The library itself** - the Python package
2. **Language models** - pre-trained statistical models that power the NLP features

### Available English Models

| Model | Size | Vectors | Accuracy | Use Case |
|-------|------|---------|----------|----------|
| `en_core_web_sm` | ~12 MB | No real vectors | Good | Prototyping, lightweight apps |
| `en_core_web_md` | ~40 MB | 20k word vectors | Better | Similarity, general use |
| `en_core_web_lg` | ~560 MB | 685k word vectors | Best | Production, high accuracy |
| `en_core_web_trf` | ~440 MB | Transformer-based | Highest | Max accuracy, GPU recommended |

In [ ]:
# Step 1: Install spaCy (uncomment the line below if not installed)
# !pip install spacy

# Step 2: Download a language model (uncomment one)
# !python -m spacy download en_core_web_sm    # Small - good for this tutorial
# !python -m spacy download en_core_web_md    # Medium - needed for word vectors (Section 10)

import spacy

print(f"spaCy version: {spacy.__version__}")

In [ ]:
# Load the English language model.
# This returns a Language object - the central processing pipeline.
# By convention, we call it "nlp".
nlp = spacy.load("en_core_web_sm")

# What components does this pipeline have?
print(f"Pipeline components: {nlp.pipe_names}")
print(f"\nDetailed pipeline info:")
for name, component in nlp.pipeline:
    print(f"  {name:>12} -> {type(component).__name__}")

## 2. Core Concepts: Doc, Token, and Span

Before diving into features, let's understand spaCy's three fundamental objects:

- **`Doc`** - A processed document (the whole text). Created by calling `nlp(text)`.
- **`Token`** - A single word/punctuation mark. Access via indexing: `doc[0]`.
- **`Span`** - A slice of the document (multiple tokens). Access via slicing: `doc[2:5]`.

These are **not** plain Python strings -- they're rich objects with linguistic annotations attached.

In [ ]:
# Process some text - this runs the ENTIRE pipeline (tokenizer, tagger, parser, NER, etc.)
doc = nlp("Apple is looking at buying U.K. startup for $1 billion.")

# Doc: the full processed text
print(f"Doc type:   {type(doc)}")
print(f"Doc text:   '{doc.text}'")
print(f"Doc length: {len(doc)} tokens")
print()

# Token: a single element
token = doc[0]
print(f"Token type:  {type(token)}")
print(f"Token text:  '{token.text}'")
print(f"Token POS:   {token.pos_}")
print()

# Span: a slice of the Doc
span = doc[1:4]  # "is looking at"
print(f"Span type:   {type(span)}")
print(f"Span text:   '{span.text}'")
print(f"Span length: {len(span)} tokens")

In [ ]:
# IMPORTANT: Doc, Token, and Span are views into the same data.
# They don't copy data -- they reference the original Doc.
# This makes spaCy very memory-efficient.

# You can iterate over a Doc just like a list:
for token in doc:
    pass  # each token is a Token object

# You can check membership:
print(f"'Apple' is in doc: {doc[0].text == 'Apple'}")

# Spans can also be created from entity detection or noun chunks (we'll see this later)
print(f"\nEntities (Spans): {doc.ents}")
print(f"Noun chunks (Spans): {list(doc.noun_chunks)}")

---
# Core NLP Features
---

## 3. Tokenization

Tokenization is the **first and most fundamental** step in NLP: splitting raw text into meaningful pieces (tokens).

spaCy's tokenizer is rule-based and handles tricky cases like:
- Contractions: `"don't"` -> `["do", "n't"]`
- URLs, emails, and special characters
- Punctuation attached to words: `"end."` -> `["end", "."]`
- Abbreviations: `"U.K."` stays as one token

In [ ]:
text = "Apple isn't looking at buying U.K. startups for $1 billion — that's fake news!"
doc = nlp(text)

# Each token has many useful attributes
print(f"{'#':<4} {'Token':<12} {'Lemma':<12} {'POS':<6} {'Alpha':<6} {'Stop':<6} {'Punct'}")
print("=" * 60)
for i, token in enumerate(doc):
    print(
        f"{i:<4} {token.text:<12} {token.lemma_:<12} {token.pos_:<6} "
        f"{str(token.is_alpha):<6} {str(token.is_stop):<6} {token.is_punct}"
    )

In [ ]:
# Notice how spaCy handles contractions:
doc = nlp("I can't believe they wouldn't've done that.")

print("Contraction handling:")
for token in doc:
    print(f"  '{token.text}' (lemma: '{token.lemma_}')")

In [ ]:
# Token shape_ attribute: a compact representation of the word's "shape"
# Uppercase = X, lowercase = x, digit = d, punctuation kept as-is
doc = nlp("Apple sold 5,000 iPhones in NYC on 2024-01-15.")

print(f"{'Token':<15} {'Shape':<15} {'Like_num':<10} {'Like_email':<12} {'Like_url'}")
print("=" * 60)
for token in doc:
    print(
        f"{token.text:<15} {token.shape_:<15} {str(token.like_num):<10} "
        f"{str(token.like_email):<12} {token.like_url}"
    )

## 4. Part-of-Speech (POS) Tagging

POS tagging labels each token with its grammatical role. spaCy provides two levels:

- **Coarse POS** (`token.pos_`): Universal categories like `NOUN`, `VERB`, `ADJ` (15 tags)
- **Fine-grained POS** (`token.tag_`): Detailed tags like `NN` (singular noun), `NNS` (plural noun), `VBG` (verb, gerund) (50+ tags)

This is useful for text analysis, grammar checking, and as features for downstream ML tasks.

In [ ]:
text = "The quick brown fox quickly jumps over two lazy dogs sitting by the river."
doc = nlp(text)

print(f"{'Token':<12} {'POS':<8} {'Fine Tag':<10} {'Explanation'}")
print("=" * 65)
for token in doc:
    print(f"{token.text:<12} {token.pos_:<8} {token.tag_:<10} {spacy.explain(token.tag_)}")

In [ ]:
# Practical example: extract specific word types
text = (
    "The brilliant scientist carefully analyzed the complex data "
    "and quickly published groundbreaking results in prestigious journals."
)
doc = nlp(text)

nouns = [token.text for token in doc if token.pos_ == "NOUN"]
verbs = [token.text for token in doc if token.pos_ == "VERB"]
adjectives = [token.text for token in doc if token.pos_ == "ADJ"]
adverbs = [token.text for token in doc if token.pos_ == "ADV"]

print(f"Nouns:      {nouns}")
print(f"Verbs:      {verbs}")
print(f"Adjectives: {adjectives}")
print(f"Adverbs:    {adverbs}")

In [ ]:
# POS distribution - useful for text analysis
from collections import Counter

text = """
Machine learning is transforming industries worldwide. Companies are investing
heavily in artificial intelligence research. New algorithms continuously improve
automated decision-making processes in healthcare, finance, and transportation.
"""
doc = nlp(text)

pos_counts = Counter(token.pos_ for token in doc if not token.is_punct and not token.is_space)

print("POS Distribution:")
for pos, count in pos_counts.most_common():
    bar = "#" * count
    print(f"  {pos:<6} {count:>2} {bar}")

## 5. Lemmatization

Lemmatization reduces words to their **base dictionary form** (called the *lemma*).

| Word | Lemma | Why? |
|------|-------|------|
| running | run | verb conjugation |
| better | well | irregular comparison |
| mice | mouse | irregular plural |
| was | be | irregular verb |

This is more sophisticated than **stemming** (which just chops off endings). Lemmatization uses the word's POS tag and a dictionary to find the true base form.

In [ ]:
text = "The striped bats were hanging on their feet and eating the best fishes. The geese were running."
doc = nlp(text)

print(f"{'Token':<12} {'Lemma':<12} {'POS':<6} {'Changed?'}")
print("=" * 42)
for token in doc:
    changed = "  <--" if token.text.lower() != token.lemma_ else ""
    print(f"{token.text:<12} {token.lemma_:<12} {token.pos_:<6} {changed}")

In [ ]:
# Practical use case: normalize text for search or comparison
def get_lemmatized_tokens(text):
    """Return a list of lemmatized, lowercased, non-stop, non-punct tokens."""
    doc = nlp(text)
    return [token.lemma_.lower() for token in doc if not token.is_stop and not token.is_punct]

# These two sentences have different surface forms but similar meaning
s1 = "The dogs were barking loudly at the running cats."
s2 = "A dog barks loud at a cat that runs."

lemmas1 = get_lemmatized_tokens(s1)
lemmas2 = get_lemmatized_tokens(s2)

print(f"Sentence 1: {s1}")
print(f"  Lemmas: {lemmas1}")
print(f"\nSentence 2: {s2}")
print(f"  Lemmas: {lemmas2}")
print(f"\nOverlap: {set(lemmas1) & set(lemmas2)}")

## 6. Named Entity Recognition (NER)

NER finds and classifies **real-world entities** in text -- people, companies, places, dates, monetary values, and more.

### Common Entity Labels

| Label | Description | Example |
|-------|-------------|--------|
| `PERSON` | People (real or fictional) | *Albert Einstein* |
| `ORG` | Companies, agencies, institutions | *Google*, *NASA* |
| `GPE` | Countries, cities, states | *France*, *New York* |
| `DATE` | Dates or periods | *January 2024*, *last week* |
| `MONEY` | Monetary values | *$1 million* |
| `PRODUCT` | Products (vehicles, software, etc.) | *iPhone*, *Windows* |
| `EVENT` | Named events | *World War II*, *Olympics* |
| `LOC` | Non-GPE locations | *the Atlantic Ocean*, *Mount Everest* |

In [ ]:
text = (
    "Elon Musk founded SpaceX in 2002 and Tesla Motors in 2003. "
    "SpaceX is headquartered in Hawthorne, California. "
    "NASA awarded SpaceX a $2.9 billion contract in April 2021 "
    "for the Artemis lunar program. Musk also acquired Twitter "
    "for approximately $44 billion in October 2022."
)
doc = nlp(text)

print(f"{'Entity':<25} {'Label':<10} {'Description'}")
print("=" * 65)
for ent in doc.ents:
    print(f"{ent.text:<25} {ent.label_:<10} {spacy.explain(ent.label_)}")

In [ ]:
# Visualize entities inline -- this renders beautifully in Jupyter
from spacy import displacy

displacy.render(doc, style="ent", jupyter=True)

In [ ]:
# You can also customize which entities to display
displacy.render(
    doc,
    style="ent",
    jupyter=True,
    options={"ents": ["PERSON", "ORG", "MONEY"], "colors": {"MONEY": "#85C1E9"}},
)

In [ ]:
# Practical: extract structured data from unstructured text
from collections import defaultdict

articles = [
    "Apple CEO Tim Cook announced new products at the Cupertino headquarters on Tuesday.",
    "Microsoft invested $10 billion in OpenAI, based in San Francisco.",
    "Google DeepMind in London published results about Gemini in December 2023.",
]

for article in articles:
    doc = nlp(article)
    entities = defaultdict(list)
    for ent in doc.ents:
        entities[ent.label_].append(ent.text)

    print(f"Text: \"{article[:60]}...\"")
    for label, values in entities.items():
        print(f"  {label}: {values}")
    print()

## 7. Dependency Parsing

Dependency parsing determines the **grammatical structure** of a sentence: which word depends on which.

Every token (except the root) has a **head** (the word it depends on) and a **dependency label** explaining the relationship.

Common dependency labels:
- `nsubj` - nominal subject ("**She** runs")
- `dobj` / `obj` - direct object ("She reads **books**")
- `amod` - adjectival modifier ("the **red** car")
- `prep` - preposition ("went **to** school")
- `ROOT` - the main verb of the sentence

In [ ]:
text = "The cat sat on the mat and watched the birds."
doc = nlp(text)

print(f"{'Token':<10} {'Dep':<12} {'Head':<10} {'Head POS':<10} {'Children'}")
print("=" * 65)
for token in doc:
    children = [child.text for child in token.children]
    print(f"{token.text:<10} {token.dep_:<12} {token.head.text:<10} {token.head.pos_:<10} {children}")

In [ ]:
# Visualize the dependency tree - very useful for understanding sentence structure
displacy.render(doc, style="dep", jupyter=True, options={"compact": True, "distance": 90})

In [ ]:
# Noun chunks: base noun phrases detected from the parse tree
# These are very useful for information extraction
doc = nlp(
    "The young data scientist from Stanford developed an innovative "
    "machine learning algorithm for natural language processing tasks."
)

print("Noun chunks (base noun phrases):")
print(f"{'Chunk':<35} {'Root':<12} {'Root Dep':<10} {'Root Head'}")
print("=" * 70)
for chunk in doc.noun_chunks:
    print(f"{chunk.text:<35} {chunk.root.text:<12} {chunk.root.dep_:<10} {chunk.root.head.text}")

In [ ]:
# Navigate the tree: find subjects and objects of verbs
doc = nlp("The engineer designed a bridge and the architect reviewed the plans.")

print("Subject-Verb-Object triples:")
for token in doc:
    if token.dep_ == "ROOT" or token.pos_ == "VERB":
        subjects = [child.text for child in token.children if "subj" in child.dep_]
        objects = [child.text for child in token.children if "obj" in child.dep_]
        if subjects or objects:
            print(f"  Verb: '{token.text}' | Subject: {subjects} | Object: {objects}")

In [ ]:
# Walk up or down the tree using .ancestors and .subtree
doc = nlp("The big red ball rolled down the steep green hill.")
token = doc[3]  # "ball"

print(f"Token: '{token.text}'")
print(f"Ancestors (path to root): {[t.text for t in token.ancestors]}")
print(f"Subtree (all descendants): {[t.text for t in token.subtree]}")
print(f"Left edge:  '{token.left_edge.text}'")
print(f"Right edge: '{token.right_edge.text}'")

## 8. Sentence Segmentation

spaCy uses the dependency parse to intelligently split text into sentences. This is more accurate than simple period-splitting because it handles abbreviations, decimal numbers, etc.

In [ ]:
text = (
    "Dr. Smith earned $1.5 million in 2023. That's impressive! "
    "However, Ms. Johnson at U.K.-based Corp. earned more. "
    "Can you believe it? I certainly can't."
)
doc = nlp(text)

print(f"Found {len(list(doc.sents))} sentences:\n")
for i, sent in enumerate(doc.sents, 1):
    print(f"  {i}. [{len(sent)} tokens] {sent.text}")

In [ ]:
# Each sentence is a Span, so you can do NLP on individual sentences
doc = nlp(
    "Barack Obama was born in Hawaii. "
    "He served as the 44th president. "
    "He left office in January 2017."
)

for i, sent in enumerate(doc.sents, 1):
    entities = [(ent.text, ent.label_) for ent in sent.ents]
    print(f"Sentence {i}: \"{sent.text}\"")
    print(f"  Entities: {entities}\n")

## 9. Morphological Analysis

Morphology goes deeper than POS tagging -- it describes the **grammatical features** of each word: tense, number, person, case, mood, etc.

In [ ]:
doc = nlp("She was reading the books that he had carefully written.")

print(f"{'Token':<12} {'POS':<6} {'Morph Features'}")
print("=" * 65)
for token in doc:
    print(f"{token.text:<12} {token.pos_:<6} {token.morph}")

In [ ]:
# Query specific morphological features
doc = nlp("I am running and they were swimming. She runs every morning.")

for token in doc:
    if token.pos_ == "VERB" or token.pos_ == "AUX":
        tense = token.morph.get("Tense")
        number = token.morph.get("Number")
        person = token.morph.get("Person")
        verb_form = token.morph.get("VerbForm")
        print(
            f"  '{token.text}' -> Tense: {tense}, Number: {number}, "
            f"Person: {person}, VerbForm: {verb_form}"
        )

---
# Intermediate Features
---

## 10. Word Vectors & Similarity

Word vectors (embeddings) represent words as dense numerical arrays. Words with similar meanings have similar vectors. This lets spaCy compute **semantic similarity** between words, spans, and documents.

> **Important:** The small model (`en_core_web_sm`) only has context vectors, not full word vectors. For meaningful similarity scores, install `en_core_web_md` or `en_core_web_lg`.
>
> ```bash
> python -m spacy download en_core_web_md
> ```

In [ ]:
# Document-level similarity: compares the average of all word vectors
doc1 = nlp("I enjoy playing football on weekends.")
doc2 = nlp("Soccer is my favorite weekend sport.")
doc3 = nlp("The stock market crashed yesterday.")

print("Document Similarity (0 = unrelated, 1 = identical):")
print(f"  'football weekends' vs 'soccer weekend sport':  {doc1.similarity(doc2):.4f}")
print(f"  'football weekends' vs 'stock market crashed':  {doc1.similarity(doc3):.4f}")
print(f"  'soccer sport'      vs 'stock market crashed':  {doc2.similarity(doc3):.4f}")

In [ ]:
# Token-level similarity matrix
doc = nlp("dog cat banana car")
tokens = list(doc)

print("Token Similarity Matrix:")
print(f"{'':>10}", end="")
for t in tokens:
    print(f"{t.text:>10}", end="")
print()
print("-" * 50)

for t1 in tokens:
    print(f"{t1.text:>10}", end="")
    for t2 in tokens:
        sim = t1.similarity(t2)
        print(f"{sim:>10.4f}", end="")
    print()

In [ ]:
# Inspecting the actual vector of a token
token = nlp("computer")[0]

print(f"Token:       '{token.text}'")
print(f"Has vector:  {token.has_vector}")
print(f"Vector norm: {token.vector_norm:.4f}")
print(f"Vector dim:  {token.vector.shape}")
print(f"First 10 values: {token.vector[:10]}")

## 11. Rule-Based Matching

spaCy's matchers let you find patterns in text based on **linguistic attributes**, not just raw text. Think of it as regex on steroids -- you can match by POS, shape, lemma, entity type, and more.

spaCy offers three matchers:
- **`Matcher`** - Token-level pattern matching with linguistic features
- **`PhraseMatcher`** - Fast exact phrase matching
- **`DependencyMatcher`** - Match based on dependency tree structure

In [ ]:
from spacy.matcher import Matcher

matcher = Matcher(nlp.vocab)

# --- Pattern 1: Adjective + Noun(s) ---
# Matches: "quick fox", "lazy dog", "tall green tree"
pattern_adj_noun = [{"POS": "ADJ"}, {"POS": "NOUN", "OP": "+"}]
matcher.add("ADJ_NOUN", [pattern_adj_noun])

# --- Pattern 2: Verb + Preposition + Noun ---
# Matches: "went to school", "looked at stars"
pattern_verb_prep = [{"POS": "VERB"}, {"POS": "ADP"}, {"POS": "NOUN"}]
matcher.add("VERB_PREP_NOUN", [pattern_verb_prep])

doc = nlp("The quick brown fox jumped over the lazy dog and ran to the tall green tree.")
matches = matcher(doc)

print(f"Found {len(matches)} matches:\n")
for match_id, start, end in matches:
    rule_name = nlp.vocab.strings[match_id]
    span = doc[start:end]
    print(f"  [{rule_name}] '{span.text}' (tokens {start}:{end})")

In [ ]:
# Quantifier operators (similar to regex)
# OP: "!"  = never (0 matches)
# OP: "?"  = optional (0 or 1)
# OP: "+"  = one or more
# OP: "*"  = zero or more

matcher2 = Matcher(nlp.vocab)

# Match phone-number-like patterns: digits separated by hyphens
phone_pattern = [
    {"SHAPE": "ddd"},
    {"TEXT": "-"},
    {"SHAPE": "ddd"},
    {"TEXT": "-"},
    {"SHAPE": "dddd"},
]
matcher2.add("PHONE_NUMBER", [phone_pattern])

# Match email-like patterns: word @ word . word
email_pattern = [
    {"LIKE_EMAIL": True},
]
matcher2.add("EMAIL", [email_pattern])

doc = nlp("Call me at 555-123-4567 or email john@example.com for details.")
matches = matcher2(doc)

for match_id, start, end in matches:
    rule = nlp.vocab.strings[match_id]
    print(f"  [{rule}] '{doc[start:end].text}'")

In [ ]:
from spacy.matcher import PhraseMatcher

# PhraseMatcher is MUCH faster for matching exact phrases.
# It uses spaCy's internal data structures instead of iterating over patterns.

phrase_matcher = PhraseMatcher(nlp.vocab, attr="LOWER")  # case-insensitive matching

technologies = [
    "machine learning", "deep learning", "natural language processing",
    "computer vision", "reinforcement learning", "neural network",
]
patterns = [nlp.make_doc(tech) for tech in technologies]
phrase_matcher.add("AI_TECH", patterns)

text = (
    "The course covers Machine Learning and Deep Learning fundamentals. "
    "Students build a Neural Network for Computer Vision, then explore "
    "Natural Language Processing and Reinforcement Learning applications."
)
doc = nlp(text)
matches = phrase_matcher(doc)

print(f"AI technologies mentioned ({len(matches)} found):")
for match_id, start, end in matches:
    print(f"  - {doc[start:end].text}")

In [ ]:
from spacy.matcher import DependencyMatcher

# DependencyMatcher: match patterns in the dependency tree
# This lets you find syntactic structures regardless of word order

dep_matcher = DependencyMatcher(nlp.vocab)

# Pattern: find "subject + verb + object" triples
pattern = [
    {"RIGHT_ID": "verb", "RIGHT_ATTRS": {"POS": "VERB"}},
    {"LEFT_ID": "verb", "REL_OP": ">", "RIGHT_ID": "subject", "RIGHT_ATTRS": {"DEP": "nsubj"}},
    {"LEFT_ID": "verb", "REL_OP": ">", "RIGHT_ID": "object", "RIGHT_ATTRS": {"DEP": "dobj"}},
]
dep_matcher.add("SVO", [pattern])

doc = nlp("The researcher published a paper and the student presented the results.")
matches = dep_matcher(doc)

print("Subject-Verb-Object triples found:")
for match_id, token_ids in matches:
    verb = doc[token_ids[0]]
    subject = doc[token_ids[1]]
    obj = doc[token_ids[2]]
    print(f"  {subject.text} -> {verb.text} -> {obj.text}")

## 12. EntityRuler - Custom NER Rules

The **EntityRuler** lets you add custom entities that the statistical model might miss. It's perfect for domain-specific terms.

In [ ]:
from spacy.language import Language

# Create a fresh pipeline so we don't conflict with earlier modifications
nlp_ruler = spacy.load("en_core_web_sm")

# Add an EntityRuler BEFORE the NER component
ruler = nlp_ruler.add_pipe("entity_ruler", before="ner")

# Define custom entity patterns
patterns = [
    # Exact phrase matches
    {"label": "PROGRAMMING_LANG", "pattern": "Python"},
    {"label": "PROGRAMMING_LANG", "pattern": "JavaScript"},
    {"label": "PROGRAMMING_LANG", "pattern": "Rust"},
    {"label": "FRAMEWORK", "pattern": "spaCy"},
    {"label": "FRAMEWORK", "pattern": "TensorFlow"},
    {"label": "FRAMEWORK", "pattern": "PyTorch"},
    # Token-level patterns (like Matcher)
    {"label": "FRAMEWORK", "pattern": [{"LOWER": "scikit"}, {"TEXT": "-"}, {"LOWER": "learn"}]},
    {"label": "FRAMEWORK", "pattern": [{"LOWER": "hugging"}, {"LOWER": "face"}]},
]
ruler.add_patterns(patterns)

doc = nlp_ruler(
    "We use Python and JavaScript for web development. "
    "For ML, we prefer PyTorch and scikit-learn over TensorFlow. "
    "Hugging Face provides great NLP models. Rust is gaining popularity."
)

print(f"{'Entity':<20} {'Label':<20}")
print("=" * 40)
for ent in doc.ents:
    print(f"{ent.text:<20} {ent.label_:<20}")

## 13. Stop Words & Text Preprocessing

**Stop words** are common words ("the", "is", "at", "and") that usually don't carry much meaning. spaCy has a built-in stop word list you can use for filtering.

In [ ]:
from spacy.lang.en.stop_words import STOP_WORDS

print(f"spaCy has {len(STOP_WORDS)} English stop words.")
print(f"\nSample stop words: {sorted(list(STOP_WORDS))[:25]}")

In [ ]:
# Filter stop words in practice
text = "The quick brown fox jumps over the lazy dog in the big park."
doc = nlp(text)

# Method 1: Using token.is_stop
filtered = [token.text for token in doc if not token.is_stop and not token.is_punct]
print(f"Original:  {text}")
print(f"Filtered:  {' '.join(filtered)}")

In [ ]:
# Customize the stop word list
# Add a custom stop word
nlp.vocab["data"].is_stop = True

# Remove a word from stop words
nlp.vocab["not"].is_stop = False

doc = nlp("The data is not clean and the results are not reliable.")
filtered = [token.text for token in doc if not token.is_stop and not token.is_punct]
print(f"Custom filtered: {' '.join(filtered)}")

# Reset changes for the rest of the notebook
nlp.vocab["data"].is_stop = False
nlp.vocab["not"].is_stop = True

In [ ]:
# Complete preprocessing pipeline for downstream tasks
def preprocess(text, allowed_pos=None):
    """Clean and normalize text for NLP tasks.

    Args:
        text: Raw input text.
        allowed_pos: Optional set of POS tags to keep (e.g., {"NOUN", "VERB", "ADJ"}).

    Returns:
        List of cleaned, lemmatized tokens.
    """
    doc = nlp(text)
    tokens = []
    for token in doc:
        # Skip stop words, punctuation, spaces, and short tokens
        if token.is_stop or token.is_punct or token.is_space or len(token) < 2:
            continue
        # Optionally filter by POS
        if allowed_pos and token.pos_ not in allowed_pos:
            continue
        tokens.append(token.lemma_.lower().strip())
    return tokens


raw = "The researchers were analyzing 5 different datasets and publishing their amazing results in 2024!"

print(f"Raw text:   {raw}")
print(f"All tokens: {preprocess(raw)}")
print(f"Nouns only: {preprocess(raw, allowed_pos={'NOUN'})}")
print(f"Verbs only: {preprocess(raw, allowed_pos={'VERB'})}")

## 14. Vocab, Lexemes & StringStore

Under the hood, spaCy converts all strings to **integer hash IDs** for performance. Understanding this helps you work more efficiently.

- **`Vocab`** - The shared vocabulary that maps strings to hash IDs
- **`Lexeme`** - An entry in the vocabulary (language-level info, not context-specific)
- **`StringStore`** - The bidirectional string-to-hash lookup table

In [ ]:
# Every string in spaCy has a unique integer hash
doc = nlp("coffee")

# StringStore: convert between strings and hashes
word = "coffee"
word_hash = nlp.vocab.strings[word]
word_back = nlp.vocab.strings[word_hash]

print(f"String -> Hash: '{word}' -> {word_hash}")
print(f"Hash -> String: {word_hash} -> '{word_back}'")

In [ ]:
# Lexemes: vocabulary entries with language-level (not context-dependent) info
# A Lexeme knows if a word is alphabetic, a digit, a stop word, etc.
# But it does NOT know POS, dependency, or entity info (those depend on context).

for word in ["apple", "123", "$", "running", "!"]:
    lexeme = nlp.vocab[word]
    print(
        f"  '{word:<10}' -> is_alpha: {lexeme.is_alpha:<6} "
        f"is_digit: {lexeme.is_digit:<6} "
        f"is_stop: {lexeme.is_stop:<6} "
        f"like_num: {lexeme.like_num}"
    )

---
# Advanced Features
---

## 15. Processing Pipelines & Performance

When you call `nlp(text)`, the text flows through a **pipeline** of components. Understanding this pipeline is key to optimizing performance.

```
Text -> Tokenizer -> Tagger -> Parser -> NER -> ... -> Doc
```

You can **disable** components you don't need to speed things up.

In [ ]:
import time

text = "spaCy is an amazing library for natural language processing." * 100

# Full pipeline
start = time.perf_counter()
doc = nlp(text)
full_time = time.perf_counter() - start

# Disable components you don't need (much faster)
start = time.perf_counter()
doc = nlp(text, disable=["ner", "parser", "lemmatizer"])
partial_time = time.perf_counter() - start

print(f"Full pipeline:    {full_time*1000:.2f} ms")
print(f"Tagger only:      {partial_time*1000:.2f} ms")
print(f"Speedup:          {full_time/partial_time:.1f}x")

In [ ]:
# nlp.pipe(): process MULTIPLE texts efficiently using batching
# This is MUCH faster than calling nlp() in a loop

texts = [
    "Apple reported strong quarterly earnings on Tuesday.",
    "Google announced a new AI research lab in Paris.",
    "Amazon acquired a robotics startup for $1.7 billion.",
    "Microsoft released a new version of its cloud platform.",
    "Meta unveiled a new virtual reality headset at CES.",
] * 20  # 100 texts

# Slow: calling nlp() in a loop
start = time.perf_counter()
docs_slow = [nlp(text) for text in texts]
loop_time = time.perf_counter() - start

# Fast: using nlp.pipe() with batching
start = time.perf_counter()
docs_fast = list(nlp.pipe(texts, batch_size=50))
pipe_time = time.perf_counter() - start

print(f"Loop (nlp() x{len(texts)}):  {loop_time*1000:.1f} ms")
print(f"Pipe (batched):         {pipe_time*1000:.1f} ms")
print(f"Speedup:                {loop_time/pipe_time:.1f}x")

In [ ]:
# nlp.make_doc(): ONLY tokenize, skip all pipeline components
# Useful when you just need tokens and nothing else

start = time.perf_counter()
for t in texts:
    doc = nlp.make_doc(t)
tokenize_time = time.perf_counter() - start

print(f"make_doc (tokenize only): {tokenize_time*1000:.2f} ms for {len(texts)} texts")
print(f"vs full pipeline:         {loop_time*1000:.2f} ms")
print(f"Speedup:                  {loop_time/tokenize_time:.0f}x")

## 16. Custom Extensions

You can attach your own **custom attributes** to Doc, Token, and Span objects. There are three types:

1. **Attribute extensions** - Simple stored values (with a default)
2. **Property extensions** - Computed via a getter (and optional setter)
3. **Method extensions** - Callable functions

In [ ]:
from spacy.tokens import Doc, Token, Span

# --- Token Extension: check if a token is a common English greeting ---
greetings = {"hello", "hi", "hey", "greetings", "howdy", "hola"}

if not Token.has_extension("is_greeting"):
    Token.set_extension("is_greeting", getter=lambda token: token.lower_ in greetings)

doc = nlp("Hello everyone, welcome to the tutorial. Hey there!")
for token in doc:
    if token._.is_greeting:
        print(f"  Greeting found: '{token.text}' at position {token.i}")

In [ ]:
# --- Doc Extension: readability score (simple metric based on word length) ---


def compute_avg_word_length(doc):
    """Compute average word length, excluding punctuation and spaces."""
    words = [token for token in doc if not token.is_punct and not token.is_space]
    if not words:
        return 0.0
    return sum(len(w.text) for w in words) / len(words)


if not Doc.has_extension("avg_word_length"):
    Doc.set_extension("avg_word_length", getter=compute_avg_word_length)

simple = nlp("The cat sat on the mat.")
complex_text = nlp("The sophisticated infrastructure demonstrated unprecedented computational efficiency.")

print(f"Simple text avg word length:  {simple._.avg_word_length:.2f} chars")
print(f"Complex text avg word length: {complex_text._.avg_word_length:.2f} chars")

In [ ]:
# --- Span Extension: method to get a span's entity breakdown ---


def get_entity_summary(span):
    """Return a dict of entity labels and their texts within this span."""
    from collections import defaultdict

    entities = defaultdict(list)
    for ent in span.ents:
        entities[ent.label_].append(ent.text)
    return dict(entities)


if not Span.has_extension("entity_summary"):
    Span.set_extension("entity_summary", method=get_entity_summary)

doc = nlp(
    "Tim Cook leads Apple in Cupertino. "
    "Satya Nadella leads Microsoft in Redmond."
)

for sent in doc.sents:
    print(f"Sentence: \"{sent.text}\"")
    print(f"  Entities: {sent._.entity_summary()}\n")

## 17. Custom Pipeline Components

You can add your own processing step to spaCy's pipeline. Every time you call `nlp(text)`, your custom component runs automatically.

There are two ways to create components:
- **`@Language.component`** - Simple function that takes a Doc and returns a Doc
- **`@Language.factory`** - A class/function that takes configuration, useful for reusable components

In [ ]:
from spacy.language import Language
from spacy.tokens import Doc


# --- Simple component: document statistics ---
@Language.component("doc_stats")
def doc_stats_component(doc):
    """Attach word count, sentence count, and entity count to the Doc."""
    doc._.word_count = sum(1 for t in doc if not t.is_punct and not t.is_space)
    doc._.sent_count = len(list(doc.sents))
    doc._.ent_count = len(doc.ents)
    return doc


# Register the extension attributes
for attr in ["word_count", "sent_count", "ent_count"]:
    if not Doc.has_extension(attr):
        Doc.set_extension(attr, default=0)

# Create a fresh pipeline and add our component
nlp_stats = spacy.load("en_core_web_sm")
nlp_stats.add_pipe("doc_stats", last=True)

print(f"Pipeline: {nlp_stats.pipe_names}")

doc = nlp_stats(
    "Barack Obama was born in Honolulu, Hawaii. "
    "He graduated from Columbia University and Harvard Law School. "
    "He became the 44th President of the United States in January 2009."
)

print(f"\nDocument Statistics:")
print(f"  Words:      {doc._.word_count}")
print(f"  Sentences:  {doc._.sent_count}")
print(f"  Entities:   {doc._.ent_count}")

In [ ]:
# --- Factory component: configurable text classifier stub ---
# Factories are more powerful - they accept config and can maintain state


@Language.factory("keyword_tagger", default_config={"keywords": {}, "attr": "tag_label"})
def create_keyword_tagger(nlp, name, keywords: dict, attr: str):
    """Factory that creates a component which tags documents based on keyword presence."""
    if not Doc.has_extension(attr):
        Doc.set_extension(attr, default="unknown")

    def keyword_tagger(doc):
        text_lower = doc.text.lower()
        for label, terms in keywords.items():
            if any(term.lower() in text_lower for term in terms):
                doc._.tag_label = label
                break
        return doc

    return keyword_tagger


nlp_tag = spacy.load("en_core_web_sm")
nlp_tag.add_pipe(
    "keyword_tagger",
    config={
        "keywords": {
            "tech": ["software", "algorithm", "computer", "AI", "programming"],
            "sports": ["football", "basketball", "tennis", "athlete", "championship"],
            "finance": ["stock", "investment", "trading", "portfolio", "dividend"],
        }
    },
)

test_texts = [
    "The new AI algorithm outperformed existing software solutions.",
    "The championship game drew millions of football fans worldwide.",
    "Stock markets rallied as investment portfolios grew significantly.",
    "The weather will be sunny with light clouds tomorrow.",
]

for text in test_texts:
    doc = nlp_tag(text)
    print(f"  [{doc._.tag_label:>8}] {text[:60]}")

## 18. Retokenization & Merging

Sometimes you want to **merge multiple tokens** into one (e.g., treating "New York City" as a single token) or **split tokens** differently. spaCy supports this via `doc.retokenize()`.

In [ ]:
# Merge named entities into single tokens
doc = nlp("New York City is home to the United Nations headquarters.")

print("Before merging:")
for token in doc:
    print(f"  {token.i}: '{token.text}'")

# Merge each entity into a single token
with doc.retokenize() as retokenizer:
    for ent in doc.ents:
        retokenizer.merge(ent)

print("\nAfter merging entities:")
for token in doc:
    print(f"  {token.i}: '{token.text}'")

In [ ]:
# Merge noun chunks into single tokens
doc = nlp("The red sports car drove past the old stone bridge.")

print("Before merging:")
print(f"  Tokens: {[t.text for t in doc]}")
print(f"  Noun chunks: {[chunk.text for chunk in doc.noun_chunks]}")

# We need to collect spans first (can't iterate and modify at the same time)
spans = list(doc.noun_chunks)
with doc.retokenize() as retokenizer:
    for span in spans:
        retokenizer.merge(span)

print("\nAfter merging noun chunks:")
print(f"  Tokens: {[t.text for t in doc]}")

## 19. Serialization - Saving & Loading

You can save and load spaCy objects (Docs, Vocab, pipelines) to and from disk or bytes. This is useful for caching processed results or building reproducible pipelines.

In [ ]:
import tempfile
import os

# Serialize a Doc to bytes (useful for caching or sending over network)
doc = nlp("Apple is looking at buying U.K. startup for $1 billion.")

# To bytes (in memory)
doc_bytes = doc.to_bytes()
print(f"Serialized Doc to {len(doc_bytes)} bytes")

# Restore from bytes
new_doc = Doc(nlp.vocab).from_bytes(doc_bytes)
print(f"Restored: '{new_doc.text}'")
print(f"Entities preserved: {[(e.text, e.label_) for e in new_doc.ents]}")

In [ ]:
# Save/load a Doc to disk using DocBin (efficient storage for multiple docs)
from spacy.tokens import DocBin

texts = [
    "Apple reported record revenue in Q4 2023.",
    "Google DeepMind published groundbreaking AI research.",
    "Amazon is expanding its drone delivery program.",
]
docs = list(nlp.pipe(texts))

# Save to disk
doc_bin = DocBin()
for doc in docs:
    doc_bin.add(doc)

with tempfile.NamedTemporaryFile(suffix=".spacy", delete=False) as f:
    temp_path = f.name
    doc_bin.to_disk(temp_path)
    file_size = os.path.getsize(temp_path)

print(f"Saved {len(docs)} docs to disk ({file_size} bytes)")

# Load from disk
loaded_bin = DocBin().from_disk(temp_path)
loaded_docs = list(loaded_bin.get_docs(nlp.vocab))

print(f"Loaded {len(loaded_docs)} docs back:")
for doc in loaded_docs:
    entities = [(e.text, e.label_) for e in doc.ents]
    print(f"  '{doc.text[:50]}' -> entities: {entities}")

# Clean up
os.unlink(temp_path)

## 20. Multi-Language Support

spaCy supports **75+ languages** with varying levels of functionality. Some languages have full statistical models, while others have basic tokenization.

You can also build **multi-language pipelines** using the `xx` (multi-language) model.

In [ ]:
# List of some supported languages (no download needed for basic tokenization)
languages = {
    "en": ("English", "Hello, how are you?"),
    "de": ("German", "Hallo, wie geht es Ihnen?"),
    "fr": ("French", "Bonjour, comment allez-vous?"),
    "es": ("Spanish", "Hola, \u00bfc\u00f3mo est\u00e1s?"),
    "zh": ("Chinese", "\u4f60\u597d\uff0c\u4f60\u600e\u4e48\u6837\uff1f"),
    "ja": ("Japanese", "\u3053\u3093\u306b\u3061\u306f\u3001\u304a\u5143\u6c17\u3067\u3059\u304b\uff1f"),
}

print("Basic tokenization for different languages:\n")
for code, (name, text) in languages.items():
    try:
        lang_nlp = spacy.blank(code)
        doc = lang_nlp(text)
        tokens = [t.text for t in doc]
        print(f"  {name:<10} ({code}): {tokens}")
    except Exception as e:
        print(f"  {name:<10} ({code}): Error - {e}")

In [ ]:
# You can download and use trained models for other languages:
# !python -m spacy download de_core_news_sm   # German
# !python -m spacy download fr_core_news_sm   # French
# !python -m spacy download es_core_news_sm   # Spanish
# !python -m spacy download zh_core_web_sm    # Chinese
# !python -m spacy download ja_core_news_sm   # Japanese

# Full list of available models: https://spacy.io/models
print("To use a language model with full NLP features:")
print("  1. Download: python -m spacy download <model_name>")
print("  2. Load:     nlp = spacy.load('<model_name>')")
print("  3. Use:      doc = nlp('Your text in that language')")

---
# Wrap-Up
---

## 21. Summary & Cheat Sheet

### Core Objects

| Object | Description | Create |
|--------|-------------|--------|
| `Language (nlp)` | The processing pipeline | `spacy.load("en_core_web_sm")` |
| `Doc` | A processed document | `nlp("text")` |
| `Token` | A single token | `doc[0]` |
| `Span` | A slice of tokens | `doc[2:5]` |
| `Lexeme` | A vocabulary entry | `nlp.vocab["word"]` |

### Feature Cheat Sheet

| Feature | API | Example |
|---------|-----|--------|
| Tokenization | `doc = nlp(text)` | Split text into tokens |
| POS Tagging | `token.pos_`, `token.tag_` | `"NOUN"`, `"NN"` |
| Lemmatization | `token.lemma_` | `"running"` -> `"run"` |
| NER | `doc.ents` | `("Apple", "ORG")` |
| Dependencies | `token.dep_`, `token.head` | `"nsubj"`, parent token |
| Sentences | `doc.sents` | Iterator of Spans |
| Morphology | `token.morph` | `Tense=Past\|VerbForm=Part` |
| Similarity | `doc1.similarity(doc2)` | `0.0` to `1.0` |
| Matching | `Matcher`, `PhraseMatcher` | Pattern-based search |
| EntityRuler | `nlp.add_pipe("entity_ruler")` | Custom NER rules |
| Extensions | `Token.set_extension(...)` | Custom attributes |
| Components | `@Language.component` | Custom pipeline steps |
| Serialization | `DocBin`, `doc.to_bytes()` | Save/load documents |

### Performance Tips

1. Use `nlp.pipe(texts)` for processing multiple texts (not a loop)
2. Disable unused components: `nlp(text, disable=["ner", "parser"])`
3. Use `nlp.make_doc(text)` when you only need tokenization
4. Use `PhraseMatcher` over `Matcher` for exact phrase matching
5. Use `DocBin` to cache and reload processed documents

### Further Resources

- [spaCy 101 - Official Introduction](https://spacy.io/usage/spacy-101)
- [Full Documentation](https://spacy.io/usage)
- [Available Models](https://spacy.io/models)
- [API Reference](https://spacy.io/api)
- [Online Course (free)](https://course.spacy.io)
- [GitHub Repository](https://github.com/explosion/spaCy)